### `nn.LazyConv2d` vs `nn.Conv2d`

Both create 2D convolutional layers, but they differ in **when** the input channels are defined.

---

#### `nn.Conv2d` — Eager (explicit)
```python
nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1, 2))
```
You must specify `in_channels` **upfront**, at construction time. PyTorch immediately allocates and initializes the weights because it knows the full shape: `(out_channels, in_channels, kH, kW)`.

---

#### `nn.LazyConv2d` — Lazy (deferred)
```python
nn.LazyConv2d(1, kernel_size=(1, 2))  # No in_channels argument
```
You **skip** `in_channels`. The weight tensor is not created yet — the layer is in an *uninitialized* state. On the **first forward pass**, PyTorch inspects the input tensor, infers `in_channels` automatically, and only then initializes the weights.

### Why `.sum()` before `.backward()`?


#### The core rule
PyTorch's `.backward()` can only be called on a **scalar** (a single number). It needs one value to differentiate with respect to.

---

#### What shape is `l`?

```python
l = (Y_hat - Y) ** 2
```

`Y_hat` and `Y` are both shape `(1, 1, 6, 7)`, so `l` is also shape `(1, 1, 6, 7)` — a **tensor with 42 values**, not a scalar.

Calling `l.backward()` directly would raise an error.

---

#### What `.sum()` does

```python
l.sum().backward()
```

It collapses all 42 values into a **single scalar** (the total loss), which `.backward()` can then differentiate to compute gradients via the chain rule.

---

#### Why sum specifically, and not e.g. `.mean()`?

You could actually use `.mean()` here too — it would also produce a scalar. The difference is just **scale**:

| Choice | Effect on gradients |
|---|---|
| `.sum()` | Gradients are the full sum — larger magnitude |
| `.mean()` | Gradients are divided by N — smaller magnitude |

In this toy example it doesn't matter much, since the learning rate `lr` can compensate either way. In practice, `.mean()` is more common because the gradient magnitude stays consistent regardless of batch size.